# Data Generation Open-Meteo

## Imports

In [1]:
import os
import requests
import pandas as pd

## Load Data

### Directories

In [2]:
base_dir = os.path.join(os.getcwd(), '..')

processed_data_dir = os.path.join(base_dir, 'processed_data')
if not os.path.exists(processed_data_dir): os.makedirs(processed_data_dir)

## Load Data

In [3]:
# approximate location for Australian postcode 2076
lat = -33.72
lon = 151.12

Set the approximate latitude and longitude for Australian postcode 2076. This is the postcode used by the ausgrid dataset. If using smart-meter data from another location, change these parameters.

In [ ]:
variable_list = ['temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,direct_normal_irradiance,diffuse_radiation,sunshine_duration']

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    'latitude': lat,
    'longitude': lon,
    'start_date': '2010-07-01',
    'end_date': '2013-06-30',
    'hourly': variable_list,
    'models': 'era5',
    'timezone': 'Australia/Sydney',
    'temperature_unit': 'celsius',
    'precipitation_unit': 'mm'}

response = requests.get(url, params=params)
response.raise_for_status()

df = pd.DataFrame(response.json()['hourly'])

Request hourly historical ERA5 weather data for that location from July 2010 through June 2013, including temperature wind, solar radiation, cloud cover, and sunshine duration. It checks that the request succeeded, then converts the returned hourly data into a pandas DataFrame. Next, inspect the dataframe:

In [ ]:
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Missing values: {df.isna().sum().sum()}\n")
display(df.head())

Shape: (26304, 8)
Columns: ['time', 'temperature_2m', 'wind_speed_10m', 'shortwave_radiation', 'cloud_cover', 'direct_normal_irradiance', 'diffuse_radiation', 'sunshine_duration']
Missing values: 0



,time,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,direct_normal_irradiance,diffuse_radiation,sunshine_duration
0,2010-07-01T00:00,4.0,9.1,0.0,0,0.0,0.0,0.0
1,2010-07-01T01:00,3.6,8.3,0.0,1,0.0,0.0,0.0
2,2010-07-01T02:00,3.3,8.4,0.0,9,0.0,0.0,0.0
3,2010-07-01T03:00,2.2,8.5,0.0,22,0.0,0.0,0.0
4,2010-07-01T04:00,2.0,8.2,0.0,9,0.0,0.0,0.0


The dataset contains 26,304 hourly records from July 2010 through June 2013, with eight columns: the timestamp and seven weather variables. There are no missing values. The first rows show nighttime readings, so zero solar radiation and sunshine are expected.

## Convert datetime columns

In [ ]:
df['datetime'] = pd.to_datetime(df['time'])
df.drop(columns=['time'], inplace=True)

### Interpolate datetimes

The Open-Meteo time resolution is one hour, where the Ausgrid time resolution is thirty minutes. If using smart-meter data with a different time-resolution, this code will need to be updated.

In [ ]:
df = (
    df.set_index('datetime')
    .resample('30min')
    .interpolate()
    .reset_index(), in_place=True)